# Step 11 — Relationship Pulse Tracker
**Tough Talks · Phase 4**

Goal: prove Gemma 4 E2B (text-only) can roll up **multiple practice rounds** for one (user, counterparty) pair into a single relationship-level pulse — health trend, recurring patterns, emerging concerns, durable wins, and the single most useful next-round focus. This is the first per-relationship aggregator in the system; every prior step (5–10) operates on one conversation at a time.

The output is one structured JSON object matching `data/schemas/pulse.schema.json`:

- `person_id` — FK to PersonVault; forced from input.
- `version` — bumped on each incremental update (v1 cold-start, v2 from prior v1, etc.).
- `round_count` — derived in code from the input rounds; minimum is **2** (a single-round trajectory does not exist).
- `health_score` ∈ `[0, 1]` — point-in-time read.
- `health_trend` — enum: `improving | stable | declining | volatile | insufficient_data`.
- `trajectory_summary` — short prose, 2–4 sentences on how the relationship has moved.
- `round_summaries[]` — one compact entry per round, chronological. `round_id` and `started_at` are copied verbatim from input; only `goal_status`, `prediction_accuracy`, and `headline` come from the model.
- `recurring_patterns[]` — shapes observed in **≥2 distinct rounds**, with `evidence_round_ids` filtered to known ids; entries with `<2` valid ids are dropped silently. Accumulates across pulse versions.
- `emerging_concerns[]` — risks that surfaced in the recent 1–2 rounds; optional `round_id` validated against known ids. Rebuilt fresh each call.
- `relationship_wins[]` — durable user-side wins; accumulates across pulse versions like PersonVault's qualitative lists.
- `next_step_recommendation` — single sentence.

**Architecture** — same one-shot hybrid-runtime pattern as Steps 8 / 9 / 10:

- Runtime in `backend/core/_runtime/pulse.py`. Notebook is a thin driver.
- Single prompt-based JSON call (analytical, not multi-turn).
- Two-shot retry: greedy first, then a single light-sampling pass (`temperature=0.3, top_p=0.9, top_k=64`) on JSON / validation failure.
- `enable_thinking=True` is the **default** per the rule in `knowledge/phases/rules.md § Decoding knobs by task type` (promoted in Step 10, confirmed N=3 across premortem / debrief / aftermath). Pulse is the fourth task-family datapoint: it cross-references multiple speaker roles across multiple rounds in a single output object, which is squarely in the thinking-helps case. `max_new_tokens=4096` follows the same rule's token-budget addendum — pulse has the largest input of any Phase 4 task (PersonVault + N rendered rounds + optional prior pulse).
- System-contract enforcement in code (defence-in-depth, same shape as Step 04's `whisper_prompt=None for other`, Step 07's `persona_name` forcing, Step 08's `scenario_id` forcing, Step 09's `_APOLOGY_CUE_RE`, Step 10's `_TRANSCRIPT_META_PREFIX_RE`):
  - `person_id` forced from PersonVault input (or `cfg.person_id`), NOT trusted to the model.
  - `version` = `prior.version + 1` (or `1` cold-start); `round_count` = `len(round_summaries)`.
  - `round_summaries` always exactly one entry per input round; `round_id` and `started_at` copied verbatim from input.
  - `goal_status` defers to the round's authoritative aftermath when the aftermath has signal; model's call is consulted only on `unknown`.
  - `health_trend` enum-normalised with synonym map (`worsening` → `declining`, `unstable` → `volatile`, etc.). When every round is `goal_status=unknown` the runtime overrides whatever the model emitted to `insufficient_data`.
  - `recurring_patterns[].evidence_round_ids` filtered to known ids; entries with `<2` valid ids dropped silently.
  - `recurring_patterns` and `relationship_wins` accumulate from `prior_pulse` (deduped, capped at 8) — same pattern as PersonVault's qualitative lists. `emerging_concerns` rebuilt fresh (concerns are point-in-time, not cumulative).
  - Required string fields rejected when empty; free-text fields capped.

**What "done" looks like for this step**
1. Text-only Gemma 4 loads (`AutoModelForCausalLM` via `LoadConfig(multimodal=False)`).
2. `generate_pulse()` returns a schema-conforming dict on the hardcoded two-round input (Round 1 from a prior simulated round, Round 2 from Step 10's hardcoded inputs/outputs).
3. `round_summaries` has exactly 2 entries with `round_id` / `started_at` matching the input verbatim.
4. Every `recurring_patterns[]` entry cites ≥ 2 known `round_id`s.
5. `health_trend` is in the allowed enum; when both rounds have aftermaths, it is NOT `insufficient_data`.
6. `health_score ∈ [0, 1]`; `next_step_recommendation` is non-empty.
7. The A/B cell renders both `enable_thinking=False` and `enable_thinking=True` outputs side-by-side.
8. The incremental cell pipes the v1 pulse back in as `prior_pulse` and produces v2 with `version=2`, prior `recurring_patterns` / `relationship_wins` deduped into the v2 output.

**Note on inputs.** Jamie's PersonVault profile and the two round records (each with its own user_goal / aftermath / debrief) are hardcoded inline. Round 2's aftermath is exactly the thinking-on aftermath that came out of Step 10's final handoff cell. Round 1 is a synthetic earlier round — same Jamie, same goal-family, three failure scenarios that all materialised, lower prediction accuracy — so the pulse has a real trajectory to read across the two rounds. In production these come from `generate_aftermath()` / `generate_debrief()` / `analyze_person_vault()` on real practice rounds. Hardcoding here avoids re-paying Steps 6 / 7 / 8 / 9 / 10's combined cost on every notebook run.

In [ ]:
# ── 0. Install / upgrade dependencies ────────────────
# Text-only path — no audio libs required. Same rule as Steps 05 / 06 /
# 07 / 08 / 09 / 10: bump only transformers + accelerate on Colab / Kaggle
# (bumping torch breaks the pre-installed torchvision / CUDA pairing).
# After this first run, RESTART THE KERNEL before continuing if you
# actually upgraded transformers — the already-imported version won't
# pick up the change.

!pip install -q -U transformers accelerate

In [ ]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────
# Same shim as Steps 01–10 — auto-clones / refreshes on Colab / Kaggle
# and clears any cached `backend.*` modules so the imports below pick
# up the freshly-pulled code instead of whatever this kernel imported
# earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ── 2. Imports ──────────────────────────────
import json
from pathlib import Path

import torch

from backend.core._runtime import (
    ALLOWED_HEALTH_TRENDS,
    ALLOWED_RELATIONSHIP_TYPES,
    ALLOWED_ROUND_GOAL_STATUSES,
    DEFAULT_MODEL_ID,
    LoadConfig,
    PulseConfig,
    PulseError,
    ROUND_COUNT_MIN,
    format_persona_profile_block,
    format_prior_pulse_block,
    format_rounds_block,
    generate_pulse,
    load_model,
)

In [ ]:
# ── 3. Configuration ──────────────────────────

MODEL_ID    = DEFAULT_MODEL_ID                # google/gemma-4-E2B-it
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SCHEMA_PATH = Path(REPO_ROOT) / "data" / "schemas" / "pulse.schema.json"

# PersonVault profile for Jamie — same v2 dict as Steps 07 / 08 / 09 / 10.
JAMIE_PROFILE = {
    "person_id": "person_cf528d57b789",
    "name": "Jamie",
    "relationship_type": "colleague",
    "version": 2,
    "conversation_count": 2,
    "profile": {
        "communication_style": "defensive",
        "emotional_triggers": [
            "citing past commitments",
            "suggesting escalation when blockers are still open",
            "implying missed communication is one-sided",
            "setting hard deadlines without acknowledging blockers",
        ],
        "de_escalation_keys": [
            "explicitly disowning blame",
            "reframing as joint problem-solving",
            "acknowledging the tightness of the timeline",
            "proposing a concrete next-step the user will own",
        ],
        "common_deflections": [
            "I told you",
            "Don't blame me",
            "Don't put this on me",
            "You weren't there",
            "staging tables aren't done",
        ],
        "responds_best_to": (
            "Clear, actionable next steps tied to specific milestones. "
            "Responds well when the user accepts responsibility for "
            "communication gaps."
        ),
    },
}

# ----- Round 1 — synthetic earlier round, two weeks before Step 10's round -----
# Goal: same family as Round 2 but specifically about last week's report.
# Aftermath shape: all three predicted scenarios materialised; user got the
# commitment partially; prediction_accuracy = 0.85. Reflects a real Round-1
# shape where Jamie's resistance caught the user off-guard and the user
# apologised through the deflection rather than naming it.

ROUND_1 = {
    "round_id": "round_jamie_2026-04-30",
    "started_at": "2026-04-30T16:00:00+00:00",
    "user_goal": (
        "Get Jamie to acknowledge that the missed report deadline "
        "matters and agree on how to flag blockers earlier next sprint."
    ),
    "aftermath": {
        "goal_outcome": {
            "status": "not_achieved",
            "summary": (
                "You didn't get the acknowledgment you were after. Jamie ran "
                "the 'I told you' deflection and you apologised your way "
                "through it instead of naming what happened."
            ),
        },
        "scenario_outcomes": [
            {
                "scenario_id": 1,
                "title": "The 'I told you' blame redirect",
                "predicted_resistance_type": "deflect",
                "match_quality": "direct_hit",
                "materialized": True,
                "evidence": (
                    "Jamie opened with 'I told you the staging tables "
                    "weren't ready' on turn 1."
                ),
                "notes": (
                    "S1 landed exactly as predicted. You acknowledged the "
                    "data team delay and apologised for pushing — that's "
                    "where the conversation slipped."
                ),
            },
            {
                "scenario_id": 2,
                "title": "Counter-attack on the new deadline",
                "predicted_resistance_type": "counter_attack",
                "match_quality": "partial",
                "materialized": True,
                "evidence": (
                    "Jamie demanded you spell out the next-sprint blockers "
                    "yourself before committing to anything."
                ),
                "notes": (
                    "S2 partly landed — softer than predicted but the burden "
                    "shifted back to you on the planning step."
                ),
            },
            {
                "scenario_id": 3,
                "title": "'Don't put this on me' guilt-trip",
                "predicted_resistance_type": "guilt_trip",
                "match_quality": "direct_hit",
                "materialized": True,
                "evidence": (
                    "Jamie said 'don't put this on me, I escalated twice' "
                    "after you raised the missed-escalation point."
                ),
                "notes": (
                    "S3 hit. You backed off the original ask to soothe her, "
                    "and the acknowledgment never landed."
                ),
            },
        ],
        "unforeseen_moments": [
            {
                "kind": "risk",
                "description": (
                    "You apologised twice for the email timing — that's a "
                    "new pattern we should watch for in future rounds."
                ),
            },
        ],
        "prediction_accuracy": 0.85,
        "next_round_focus": (
            "Practice naming the deflection out loud the first time it "
            "appears, instead of apologising your way through it."
        ),
        "aftermath_id": "aftermath_demo_round_1_a1",
        "generated_at": "2026-04-30T17:00:00+00:00",
    },
    "debrief": {
        "wins": [],
        "ground_lost": [
            {
                "turn": 1,
                "quote": (
                    "Sorry, I know the data team was behind — I should have "
                    "flagged it earlier."
                ),
                "reason": (
                    "You accepted blame on Jamie's first deflection instead of "
                    "naming the pattern. That set the tone for the rest of the "
                    "conversation."
                ),
            },
            {
                "turn": 3,
                "quote": (
                    "Sorry, I shouldn't have pushed — let's just figure out "
                    "what blockers you see."
                ),
                "reason": (
                    "You folded on the original ask after Jamie's guilt-trip. "
                    "The conversation shifted to her planning the next sprint."
                ),
            },
        ],
        "over_apologies": [
            {
                "turn": 1,
                "quote": (
                    "Sorry, I know the data team was behind — I should have "
                    "flagged it earlier."
                ),
            },
            {
                "turn": 3,
                "quote": "Sorry, I shouldn't have pushed.",
            },
        ],
        "missed_openings": [],
        "one_fix_next_time": (
            "When Jamie deflects with 'I told you', name it out loud "
            "instead of apologising — that's where the conversation "
            "keeps slipping."
        ),
        "debrief_id": "debrief_demo_round_1_a1",
        "generated_at": "2026-04-30T16:50:00+00:00",
    },
}

# ----- Round 2 — Step 10's hardcoded round + the thinking-on aftermath -----
# Aftermath copied from Step 10's final handoff cell (the thinking=True
# branch). Debrief copied from Step 09's thinking-on payload. Same Jamie,
# same goal family, two weeks later — so we can see if the user's response
# improved between Round 1 and Round 2.

ROUND_2 = {
    "round_id": "round_jamie_2026-05-14",
    "started_at": "2026-05-14T15:30:00+00:00",
    "user_goal": (
        "Get Jamie to commit to delivering the staging-tables data by "
        "Wednesday EOD and to acknowledge that the escalation last week "
        "needed to land more clearly — not just be sent."
    ),
    "aftermath": {
        "goal_outcome": {
            "status": "partial",
            "summary": (
                "You successfully got Jamie to commit to a daily check-in "
                "and a new communication standard, which is a win. However, "
                "the hard Wednesday EOD commitment was not secured, and the "
                "core issue of the escalation needing to land clearly was "
                "only partially addressed."
            ),
        },
        "scenario_outcomes": [
            {
                "scenario_id": 1,
                "title": "The 'I told you' blame redirect",
                "predicted_resistance_type": "deflect",
                "match_quality": "direct_hit",
                "materialized": True,
                "evidence": (
                    "Jamie's first reply — 'I told you we were short on "
                    "time.' She ran the predicted deflection."
                ),
                "notes": (
                    "This prediction was spot on. Jamie ran the deflection "
                    "play by immediately using 'I told you' and framing the "
                    "issue as a systemic problem rather than a personal "
                    "failure. You handled this well by moving past the blame "
                    "and focusing on the next step."
                ),
            },
            {
                "scenario_id": 2,
                "title": "Counter-attack on the new deadline",
                "predicted_resistance_type": "counter_attack",
                "match_quality": "partial",
                "materialized": True,
                "evidence": (
                    "USER 2 followed up with the hard deadline ask; Jamie's "
                    "reply began 'Wednesday EOD is tight' — softer than the "
                    "counter-attack we predicted."
                ),
                "notes": (
                    "The counter-attack happened, but it was softer than "
                    "predicted. Instead of a hard demand for a list, Jamie "
                    "framed it as needing to 'manage expectations.' You "
                    "successfully held the line on the deadline while still "
                    "giving her the information she needed to feel in control."
                ),
            },
            {
                "scenario_id": 3,
                "title": "'Don't put this on me' guilt-trip",
                "predicted_resistance_type": "guilt_trip",
                "match_quality": "direct_hit",
                "materialized": True,
                "evidence": (
                    "Jamie said 'I told you, don't put this on me' when you "
                    "raised the missed-escalation point."
                ),
                "notes": (
                    "This prediction was a direct hit. Jamie used the "
                    "guilt-trip to shift the blame onto the process. You "
                    "handled this by reframing the issue as a joint problem "
                    "('our problem to solve together'), which successfully "
                    "diffused the tension."
                ),
            },
        ],
        "unforeseen_moments": [
            {
                "kind": "opportunity",
                "turn": 4,
                "description": (
                    "You introduced a concrete, joint-problem-solving "
                    "solution (the daily check-in). This was a great move "
                    "because it shifted the focus from past failures to "
                    "future collaboration, which Jamie was receptive to."
                ),
            },
            {
                "kind": "opportunity",
                "turn": 5,
                "description": (
                    "You successfully established a new communication "
                    "standard (ping vs. email). This was a positive outcome "
                    "that addresses the underlying communication gap you "
                    "were trying to fix."
                ),
            },
        ],
        "prediction_accuracy": 0.75,
        "next_round_focus": (
            "Practice handling the soft 'manage expectations' pushback — "
            "that's the real shape of Jamie's resistance, not the harder "
            "counter-attack we wrote up."
        ),
        "aftermath_id": "aftermath_demo_round_2_b2",
        "generated_at": "2026-05-14T17:00:00+00:00",
    },
    "debrief": {
        "wins": [
            {
                "turn": 2,
                "description": (
                    "You moved the focus from past blame to a concrete "
                    "shared commitment with a specific deadline."
                ),
            },
            {
                "turn": 3,
                "description": (
                    "You reframed the escalation issue as a joint problem, "
                    "directly addressing Jamie's defensive stance."
                ),
            },
            {
                "turn": 4,
                "description": (
                    "You proposed a clear, actionable next step (the daily "
                    "check-in) and took responsibility for unblocking."
                ),
            },
        ],
        "ground_lost": [],
        "over_apologies": [],
        "missed_openings": [],
        "one_fix_next_time": (
            "When you propose a solution, immediately follow it up with "
            "the specific commitment you need from the other person to "
            "make that solution happen."
        ),
        "debrief_id": "debrief_demo_round_2_b2",
        "generated_at": "2026-05-14T16:30:00+00:00",
    },
}

ROUNDS = [ROUND_1, ROUND_2]

print(f"Model              : {MODEL_ID}")
print(f"Device             : {DEVICE}")
print(f"Persona            : {JAMIE_PROFILE['name']} ({JAMIE_PROFILE['relationship_type']})")
print(f"Rounds             : {len(ROUNDS)} (min={ROUND_COUNT_MIN})")
for r in ROUNDS:
    print(
        f"  - {r['round_id']} @ {r['started_at']} "
        f"goal_status={r['aftermath']['goal_outcome']['status']} "
        f"prediction_accuracy={r['aftermath']['prediction_accuracy']:.2f}"
    )

In [ ]:
# ── 4. Preview the rendered context blocks (no model needed) ──────────
# All three rendered blocks are what the runtime feeds into the prompt.
# Rendering them here is purely diagnostic — it lets us verify the
# inputs are well-formed before paying for the model load.

print("=" * 76)
print("PERSON PROFILE BLOCK (Jamie)")
print("=" * 76)
print(format_persona_profile_block(JAMIE_PROFILE))

print()
print("=" * 76)
print("ROUNDS BLOCK (2 rounds, oldest first)")
print("=" * 76)
print(format_rounds_block(ROUNDS))

print()
print("=" * 76)
print("PRIOR PULSE BLOCK (cold-start — no prior pulse yet)")
print("=" * 76)
print(format_prior_pulse_block(None))

In [ ]:
# ── 5. Load the text-only processor + model ────────────────
# Pulse reasons over WORDS — same rationale as TalkDNA / PersonVault
# / persona-sim / premortem / debrief / aftermath. The `multimodal=False`
# (default) path loads `AutoModelForCausalLM`, which is lighter on VRAM
# and slightly faster than the multimodal class. Live Mode (Phase 5+)
# will reuse this same loaded model across components, so the load
# cost is amortised.

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

In [ ]:
# ── 6. Generate the v1 pulse (cold-start, thinking=True per the rule) ─────
# Single prompt-based JSON call — no rolling history, mirror of
# Steps 08 / 09 / 10. The runtime threads the person profile, the two
# rendered round records, and the prior-pulse sentinel into the
# `data/prompts/pulse.md` template, calls `chat()`, parses the JSON,
# validates / coerces against the schema, and returns a schema-
# conforming dict. `enable_thinking=True` is the default per the rule
# in `knowledge/phases/rules.md § Decoding knobs by task type`.

cfg = PulseConfig(
    rounds=ROUNDS,
    person_profile=JAMIE_PROFILE,
    prior_pulse=None,
    enable_thinking=True,
    max_new_tokens=4096,
)

try:
    pulse_v1 = generate_pulse(processor, model, cfg=cfg)
except PulseError as exc:
    print(f"FAILED: {exc}")
    for attempt in getattr(exc, "attempts", []):
        print(f"  attempt ({attempt['sampling']}): {attempt['error']}")
        print(f"  raw_head: {attempt['raw_head']!r}")
    raise

print("=" * 76)
print("HEALTH")
print("=" * 76)
print(f"  health_score        : {pulse_v1['health_score']:.2f}")
print(f"  health_trend        : {pulse_v1['health_trend']}")
print(f"  trajectory_summary  : {pulse_v1['trajectory_summary']}")

print()
print("=" * 76)
print(f"ROUND SUMMARIES ({pulse_v1['round_count']} — chronological)")
print("=" * 76)
for s in pulse_v1["round_summaries"]:
    print(f"\n  {s['round_id']} @ {s['started_at']}")
    print(f"    goal_status         : {s['goal_status']}")
    print(f"    prediction_accuracy : {s['prediction_accuracy']:.2f}")
    print(f"    headline            : {s['headline']}")

print()
print("=" * 76)
print(f"RECURRING PATTERNS ({len(pulse_v1['recurring_patterns'])})")
print("=" * 76)
if not pulse_v1["recurring_patterns"]:
    print("  (none)")
for entry in pulse_v1["recurring_patterns"]:
    ids_str = ", ".join(entry["evidence_round_ids"])
    print(f"  - {entry['pattern']}")
    print(f"      evidence: [{ids_str}]")

print()
print("=" * 76)
print(f"EMERGING CONCERNS ({len(pulse_v1['emerging_concerns'])})")
print("=" * 76)
if not pulse_v1["emerging_concerns"]:
    print("  (none)")
for entry in pulse_v1["emerging_concerns"]:
    rid_str = f" @{entry['round_id']}" if "round_id" in entry else ""
    print(f"  -{rid_str} {entry['concern']}")

print()
print("=" * 76)
print(f"RELATIONSHIP WINS ({len(pulse_v1['relationship_wins'])})")
print("=" * 76)
if not pulse_v1["relationship_wins"]:
    print("  (none)")
for entry in pulse_v1["relationship_wins"]:
    rid_str = f" @{entry['round_id']}" if "round_id" in entry else ""
    print(f"  -{rid_str} {entry['win']}")

print()
print(f"next_step_recommendation: {pulse_v1['next_step_recommendation']}")

In [ ]:
# ── 7. Schema validation + results table ─────────────────────
# Hand-rolled validator (no jsonschema dep — matches Steps 04 / 05 / 06
# / 07 / 08 / 09 / 10). Checks:
#   - all required top-level keys present
#   - person_id forced from PersonVault input
#   - version == 1 on cold-start
#   - round_count == len(ROUNDS); >= ROUND_COUNT_MIN
#   - round_summaries: one entry per input round, in order; round_id
#     and started_at copied verbatim from input
#   - goal_status values in allowed enum
#   - prediction_accuracy in [0, 1] on every round
#   - health_score in [0, 1]; health_trend in allowed enum
#   - recurring_patterns: every entry cites >= 2 known round_ids
#   - emerging_concerns: optional round_id (if present) is a known id
#   - relationship_wins: optional round_id (if present) is a known id
#   - relationship_type in PersonVault enum
#   - next_step_recommendation non-empty
#   - pulse_id attached with the expected prefix

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
TOP_REQUIRED = set(schema["required"])

KNOWN_ROUND_IDS = {r["round_id"] for r in ROUNDS}


def _is_non_empty_str(value) -> bool:
    return isinstance(value, str) and bool(value.strip())


summary_errors: list[str] = []
for i, s in enumerate(pulse_v1["round_summaries"]):
    expected_input = ROUNDS[i] if i < len(ROUNDS) else None
    if expected_input is not None:
        if s["round_id"] != expected_input["round_id"]:
            summary_errors.append(
                f"round_summaries[{i}]: round_id {s['round_id']!r} != "
                f"input {expected_input['round_id']!r}"
            )
        if s["started_at"] != expected_input["started_at"]:
            summary_errors.append(
                f"round_summaries[{i}]: started_at {s['started_at']!r} != "
                f"input {expected_input['started_at']!r}"
            )
    if s["goal_status"] not in ALLOWED_ROUND_GOAL_STATUSES:
        summary_errors.append(
            f"round_summaries[{i}]: goal_status {s['goal_status']!r} not in enum"
        )
    if not (0.0 <= s["prediction_accuracy"] <= 1.0):
        summary_errors.append(
            f"round_summaries[{i}]: prediction_accuracy {s['prediction_accuracy']!r} "
            f"not in [0, 1]"
        )
    if not _is_non_empty_str(s["headline"]):
        summary_errors.append(f"round_summaries[{i}]: headline empty or non-string")

pattern_errors: list[str] = []
for i, entry in enumerate(pulse_v1["recurring_patterns"]):
    if not _is_non_empty_str(entry.get("pattern")):
        pattern_errors.append(
            f"recurring_patterns[{i}]: pattern empty or non-string"
        )
    ids = entry.get("evidence_round_ids")
    if not isinstance(ids, list) or len(ids) < 2:
        pattern_errors.append(
            f"recurring_patterns[{i}]: evidence_round_ids must have >= 2 entries, "
            f"got {ids!r}"
        )
    else:
        for rid in ids:
            if rid not in KNOWN_ROUND_IDS:
                pattern_errors.append(
                    f"recurring_patterns[{i}]: evidence id {rid!r} not in "
                    f"known round_ids {sorted(KNOWN_ROUND_IDS)}"
                )

concern_errors: list[str] = []
for i, entry in enumerate(pulse_v1["emerging_concerns"]):
    if not _is_non_empty_str(entry.get("concern")):
        concern_errors.append(
            f"emerging_concerns[{i}]: concern empty or non-string"
        )
    rid = entry.get("round_id")
    if rid is not None and rid not in KNOWN_ROUND_IDS:
        concern_errors.append(
            f"emerging_concerns[{i}]: round_id {rid!r} not in known ids"
        )

win_errors: list[str] = []
for i, entry in enumerate(pulse_v1["relationship_wins"]):
    if not _is_non_empty_str(entry.get("win")):
        win_errors.append(f"relationship_wins[{i}]: win empty or non-string")
    rid = entry.get("round_id")
    if rid is not None and rid not in KNOWN_ROUND_IDS:
        win_errors.append(
            f"relationship_wins[{i}]: round_id {rid!r} not in known ids"
        )

checks: list[tuple[str, bool, str]] = [
    ("text_model_loaded",              True,                                                                                  f"{n_params:.1f}B params on {model.device}"),
    ("all_required_keys_present",      TOP_REQUIRED.issubset(pulse_v1.keys()),                                                  f"{sorted(TOP_REQUIRED & set(pulse_v1.keys()))}"),
    ("person_id_forced_from_input",    pulse_v1["person_id"] == JAMIE_PROFILE["person_id"],                                     f"{pulse_v1['person_id']}"),
    ("version_is_1_on_cold_start",     pulse_v1["version"] == 1,                                                                f"v{pulse_v1['version']}"),
    ("round_count_matches_input",      pulse_v1["round_count"] == len(ROUNDS),                                                  f"{pulse_v1['round_count']} / {len(ROUNDS)}"),
    ("round_count_at_least_min",       pulse_v1["round_count"] >= ROUND_COUNT_MIN,                                              f"min={ROUND_COUNT_MIN}"),
    ("round_summaries_in_input_order", [s["round_id"] for s in pulse_v1["round_summaries"]] == [r["round_id"] for r in ROUNDS], f"{[s['round_id'] for s in pulse_v1['round_summaries']]}"),
    ("round_summaries_clean",          not summary_errors,                                                                      f"{pulse_v1['round_count']} entries"),
    ("health_score_unit",              0.0 <= pulse_v1["health_score"] <= 1.0,                                                  f"{pulse_v1['health_score']:.2f}"),
    ("health_trend_valid",             pulse_v1["health_trend"] in ALLOWED_HEALTH_TRENDS,                                       f"{pulse_v1['health_trend']}"),
    ("trajectory_summary_non_empty",   _is_non_empty_str(pulse_v1["trajectory_summary"]),                                       f"{pulse_v1['trajectory_summary'][:80]}..."),
    ("recurring_patterns_clean",       not pattern_errors,                                                                      f"{len(pulse_v1['recurring_patterns'])} entries"),
    ("emerging_concerns_clean",        not concern_errors,                                                                      f"{len(pulse_v1['emerging_concerns'])} entries"),
    ("relationship_wins_clean",        not win_errors,                                                                          f"{len(pulse_v1['relationship_wins'])} entries"),
    ("relationship_type_valid",        pulse_v1.get("relationship_type") in ALLOWED_RELATIONSHIP_TYPES,                         f"{pulse_v1.get('relationship_type')}"),
    ("next_step_non_empty",            _is_non_empty_str(pulse_v1["next_step_recommendation"]),                                 f"{pulse_v1['next_step_recommendation'][:80]}..."),
    ("pulse_id_attached",              isinstance(pulse_v1.get("pulse_id"), str) and pulse_v1["pulse_id"].startswith("pulse_"),  f"{pulse_v1.get('pulse_id', '<missing>')}"),
]

print("=" * 76)
print("STEP 11 RESULTS — Gemma 4 Relationship Pulse Tracker")
print("=" * 76)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:36s}  {note}")
    if not ok:
        all_ok = False

for label, errs in (("round_summaries", summary_errors), ("recurring_patterns", pattern_errors), ("emerging_concerns", concern_errors), ("relationship_wins", win_errors)):
    if errs:
        print(f"\n{label} errors:")
        for line in errs:
            print(f"  - {line}")

print()
print("OVERALL:", "READY FOR PHASE 5" if all_ok else "FIX FAILURES ABOVE")

In [ ]:
# ── 8. A/B: enable_thinking=False vs enable_thinking=True (same inputs) ──
# Fourth task-family datapoint for the rule in
# `knowledge/phases/rules.md § Decoding knobs by task type`. Pulse is
# the most cross-field-consistency-heavy analytical task in Phase 4 so
# far: every recurring_patterns entry must cite >= 2 round_ids verbatim,
# every emerging_concern's optional round_id must be one of the input
# ids, health_trend must agree with the per-round goal_status arc,
# round_summaries must match the input round metadata exactly.
# Premortem (N=2), debrief (N=1), and aftermath (N=1) all showed
# thinking-on tightening this kind of cross-field consistency.
#
# Token budget for the thinking branch — 4096 by default (the rule's
# token-budget addendum: pulse has the largest input of Phase 4, so the
# trace needs more headroom than premortem's 2048). If the thinking
# branch truncates anyway, the finding is clean: pulse is too
# token-hungry at E2B scale for thinking-on, and the default should
# flip to False.

cfg_no_thinking = PulseConfig(
    rounds=ROUNDS,
    person_profile=JAMIE_PROFILE,
    prior_pulse=None,
    enable_thinking=False,
    max_new_tokens=2048,
)
cfg_thinking = PulseConfig(
    rounds=ROUNDS,
    person_profile=JAMIE_PROFILE,
    prior_pulse=None,
    enable_thinking=True,
    max_new_tokens=4096,
)

pulse_no_thinking = generate_pulse(processor, model, cfg=cfg_no_thinking)

pulse_thinking = None
thinking_err = None
try:
    pulse_thinking = generate_pulse(processor, model, cfg=cfg_thinking)
except PulseError as exc:
    thinking_err = exc


def _render_short(label: str, p: dict) -> None:
    print(f"\n--- {label} ---")
    print(f"  health_score        : {p['health_score']:.2f}")
    print(f"  health_trend        : {p['health_trend']}")
    print(f"  trajectory_summary  : {p['trajectory_summary']}")
    for s in p["round_summaries"]:
        print(
            f"  {s['round_id']}: goal_status={s['goal_status']}, "
            f"prediction_accuracy={s['prediction_accuracy']:.2f}"
        )
        print(f"      headline: {s['headline']}")
    print(f"  recurring_patterns  : {len(p['recurring_patterns'])}")
    for entry in p["recurring_patterns"]:
        ids_str = ", ".join(entry["evidence_round_ids"])
        print(f"      * {entry['pattern']}")
        print(f"        evidence: [{ids_str}]")
    print(f"  emerging_concerns   : {len(p['emerging_concerns'])}")
    for entry in p["emerging_concerns"]:
        rid_str = f" @{entry['round_id']}" if "round_id" in entry else ""
        print(f"      *{rid_str} {entry['concern']}")
    print(f"  relationship_wins   : {len(p['relationship_wins'])}")
    for entry in p["relationship_wins"]:
        rid_str = f" @{entry['round_id']}" if "round_id" in entry else ""
        print(f"      *{rid_str} {entry['win']}")
    print(f"  next_step_recommendation: {p['next_step_recommendation']}")


print("=" * 76)
print("A/B — enable_thinking on the same pulse inputs")
print("=" * 76)

_render_short("thinking=False", pulse_no_thinking)

if pulse_thinking is not None:
    _render_short("thinking=True (default)", pulse_thinking)
else:
    print("\n--- thinking=True ---")
    print(f"  FAILED: {thinking_err}")
    for attempt in getattr(thinking_err, "attempts", []):
        print(f"\n  attempt ({attempt['sampling']}): {attempt['error']}")
        print("  raw_head (first 500 chars):")
        print(f"  {attempt['raw_head']!r}")

In [ ]:
# ── 9. Incremental update: v1 → v2 (same rounds, prior_pulse fed in) ─────
# Demonstrates the incremental contract: passing the v1 pulse back in
# as `prior_pulse` must bump version 1 → 2, accumulate / dedup
# `recurring_patterns` and `relationship_wins`, and rebuild fresh
# `health_score` / `health_trend` / `trajectory_summary` /
# `emerging_concerns` / `next_step_recommendation`. Same accumulation
# pattern as PersonVault v1 → v2 in Step 06.
#
# In production this is what happens at the start of the user's next
# practice round: the v1 pulse from a prior session is loaded from
# local JSON, this notebook's call gets repeated with the new round
# appended to ROUNDS, and the pulse object is rewritten to disk.

cfg_v2 = PulseConfig(
    rounds=ROUNDS,
    person_profile=JAMIE_PROFILE,
    prior_pulse=pulse_v1,
    enable_thinking=True,
    max_new_tokens=4096,
)

pulse_v2 = generate_pulse(processor, model, cfg=cfg_v2)

v1_pattern_keys = {entry["pattern"].strip().lower() for entry in pulse_v1["recurring_patterns"]}
v2_pattern_keys = {entry["pattern"].strip().lower() for entry in pulse_v2["recurring_patterns"]}
v1_win_keys     = {entry["win"].strip().lower() for entry in pulse_v1["relationship_wins"]}
v2_win_keys     = {entry["win"].strip().lower() for entry in pulse_v2["relationship_wins"]}

patterns_dropped = v1_pattern_keys - v2_pattern_keys
wins_dropped     = v1_win_keys - v2_win_keys

print("=" * 76)
print("INCREMENTAL CHECK — v1 -> v2")
print("=" * 76)
print(f"version              : v1={pulse_v1['version']} -> v2={pulse_v2['version']}")
print(f"person_id unchanged  : {pulse_v1['person_id'] == pulse_v2['person_id']}")
print(f"recurring_patterns   : v1={len(pulse_v1['recurring_patterns'])} -> v2={len(pulse_v2['recurring_patterns'])}")
print(f"  v1 keys not in v2  : {len(patterns_dropped)} {sorted(patterns_dropped)}")
print(f"relationship_wins    : v1={len(pulse_v1['relationship_wins'])} -> v2={len(pulse_v2['relationship_wins'])}")
print(f"  v1 keys not in v2  : {len(wins_dropped)} {sorted(wins_dropped)}")
print(f"v2.health_trend      : {pulse_v2['health_trend']}")
print(f"v2.health_score      : {pulse_v2['health_score']:.2f}")
print(f"v2.next_step         : {pulse_v2['next_step_recommendation']}")

In [ ]:
# ── 10. Final pulse dict — the JSON Phase 5's API will return verbatim ──
# Phase 5 (the FastAPI service) will hand this object back to the
# frontend whenever the user opens the relationship view for Jamie.
# Phase 13's local JSON storage layer will persist it on every
# practice round. This cell prints the final v1 pulse — the cold-
# start case the frontend sees on a relationship's first roll-up.

print("=" * 76)
print("FINAL PULSE (v1 — cold-start)")
print("=" * 76)
print(json.dumps(pulse_v1, indent=2, ensure_ascii=False))